# Vizualization

# Import libraries

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
import time
import datetime
import seaborn as sns
import matplotlib.pyplot as plt
from termcolor import colored, cprint


import sys
from os import listdir
from os.path import isfile, join

import numpy as np
import pandas as pd  # Import Pandas and Numpy to create databases
import tensorflow as tf  # Import tensorflow to do tensor manipulations

from abyss.helpers import metrics_viewer
from abyss.models import create_autoencoder
from abyss.glob import glob
from sklearn.metrics import silhouette_score

In [ ]:
path_data = f"data/phenotype"
humans = pd.read_pickle(f"{path_data}/all.panel")

In [ ]:
# Parse command line arguments
args = sys.argv
# Set default values if arguments are not provided
epoch = 15
bottleneck = 2
PCs = 2
bin = 1

In [ ]:
# Define paths and directories
PATH_GENO = glob.PATH_GENO
path_output = f"{PATH_GENO}/05_Latent_populations/{bottleneck}_bottleneck_dims/"

In [ ]:
list_bins = [f for f in os.listdir(path_output) if not f.endswith("metric_plots")]

In [ ]:
mafs_to_append = []
sils_to_append = []
type_pop = []
for bin in list_bins:
    path_bin = path_output + bin
    files = os.listdir(path_bin)
    for file in files[5:8]:
        maf = float(file.split('_')[4])
        path_file = path_bin + "/" + file
        dims = pd.read_pickle(path_file)
        # Scatter plot with coloring by 'population'
        temp = dims.copy()
        temp['pop'] = list(humans['Superpopulation code'])
        temp['mini pop'] = list(humans['Population code'])
        
        # Using the encoded labels in silhouette_score
        silhouette_pop = silhouette_score(dims, temp['pop'])
        mafs_to_append.append(maf)
        sils_to_append.append(silhouette_pop)
        type_pop.append("super population")
        
        # Using the encoded labels in silhouette_score
        silhouette_pop = silhouette_score(dims, temp['mini pop'])
        mafs_to_append.append(maf)
        sils_to_append.append(silhouette_pop)
        type_pop.append("population")
        #dims['pop'] = list(humans['Superpopulation code'])
        #sns.scatterplot(data=dims, x='dim1', y='dim2', hue='pop',palette='rocket')
        #plt.xlabel('dim1')
        #plt.ylabel('dim2')
        #plt.title(f"Lower dimensionality representation with silhouette score = {silhouette_pop}")
        #plt.show()

In [ ]:
# Creating a DataFrame
df = pd.DataFrame({
    'MAF': mafs_to_append,
    'Type of population': type_pop,
    'Silhouette Score': sils_to_append
})


In [ ]:
df

In [ ]:
# Creating the scatter plot
sns.scatterplot(x='MAF', y='Silhouette Score', hue='Type of population', data=df, palette="rocket")

# Setting the title and labels for clarity
plt.title('Scatter Plot of Silhouette Scores by Bin and Dimension')
plt.xlabel('Allele frequency')
plt.ylabel('Silhouette Score')

# Display the plot
plt.show()

# Check per populations

In [ ]:
# Get the unique values from the 'Superpopulation code' column
unique_values = humans['Superpopulation code'].unique()
# Filter out the 'EUR' element
filtered_values = [value for value in unique_values if value != 'EUR,AFR']

In [ ]:
mafs_to_append = []
sils_to_append = []
type_pop = []
for pop in filtered_values:
    for bin in list_bins:
        path_bin = path_output + bin
        files = os.listdir(path_bin)
        for file in files:
            maf = float(file.split('_')[4])
            path_file = path_bin + "/" + file
            dims = pd.read_pickle(path_file)
            temp = dims.copy()
            temp['pop'] = list(humans['Superpopulation code'])
            temp['mini pop'] = list(humans['Population code'])
            
            temp = temp[temp['pop'] == pop]
            dims['pop'] = list(humans['Superpopulation code'])
            dims = dims[dims['pop'] == pop]
            # Drop the 'B' column
            dims = dims.drop('pop', axis=1)

            # Using the encoded labels in silhouette_score
            silhouette_pop = silhouette_score(dims, temp['mini pop'])
            mafs_to_append.append(maf)
            sils_to_append.append(silhouette_pop)
            type_pop.append(pop)

In [ ]:
# Creating a DataFrame
df_mini = pd.DataFrame({
    'MAF': mafs_to_append,
    'Type of population': type_pop,
    'Silhouette Score': sils_to_append
})

In [ ]:
df_mini

In [ ]:
# Creating the scatter plot
sns.scatterplot(x='MAF', y='Silhouette Score', hue='Type of population', data=df_mini, palette="rocket")

# Setting the title and labels for clarity
plt.title('Scatter Plot of Silhouette Scores by Bin and Dimension')
plt.xlabel('Allele frequency')
plt.ylabel('Silhouette Score')

# Display the plot
plt.show()

In [ ]:
dims['pop'] = temp['mini pop']

sns.scatterplot(data=dims, x='dim1', y='dim2', hue='pop',palette='rocket')
plt.xlabel('dim1')
plt.ylabel('dim2')
plt.title(f"Lower dimensionality representation with silhouette score = {silhouette_pop}")
plt.show()